# EX_08 — Introducción a agentes (ejercicios)

**Notebook de referencia:** `notebook/08_Introduccion_Agentes.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Definir 2 herramientas

Escribe funciones Python puras `get_time_utc()` (puede ser fake) y `hash_text(s: str)` (usa `hashlib.sha256` en hex). Estas serán tus "tools".


In [3]:
import hashlib
from datetime import datetime, timezone

def get_time_utc() -> str:
    # TODO: return ISO timestamp (real or mocked)
    """Devuelve la fecha y hora actual en formato UTC (ISO 8601)."""
    return datetime.now(timezone.utc).isoformat()

def hash_text(s: str) -> str:
    # TODO: Implement text hashing
    """Devuelve el hash SHA-256 de la cadena de texto proporcionada."""
    return hashlib.sha256(s.encode()).hexdigest()

get_time_utc()

'2026-06-22T14:14:20.832953+00:00'

## Actividad 2 — Cuándo usar tool

Para cada intención del usuario (`"What time is it?"`, `"Digest of hello"`), escribe en comentarios si el LLM debería llamar tool o responder directo.


In [4]:
# TODO: your comments per intent
# TODO: your comments per intent

# 1. "What time is it?"
# Veredicto: LLAMAR TOOL. 
# Por qué: El LLM no tiene acceso al reloj del sistema ni noción del tiempo real actual. Depende de una herramienta externa para consultar la hora exacta.

# 2. "Digest of hello"
# Veredicto: RESPONDER DIRECTO (o Tool si es criptografía).
# Por qué: Si "digest" significa explicar o procesar semánticamente la palabra "hello", el LLM puede responder directo usando su entrenamiento. Si se refiere a calcular un "hash digest" matemático, necesitaría una tool de calculadora/criptografía, ya que los LLM fallan en cálculos exactos.

## Actividad 3 — Bucles

En español (celda markdown), explica el riesgo de **bucles infinitos** tool→modelo→tool y una mitigación (límite de pasos, detector de repetición).


_Tu explicación:_

...


El riesgo de los bucles infinitos (Tool → Modelo → Tool)
Cuando trabajamos con Agentes de IA, el modelo tiene la autonomía de decidir cuándo usar una herramienta y cuándo detenerse. El gran riesgo ocurre cuando el modelo se "atasca" (por ejemplo, si una herramienta le devuelve un mensaje de error que no entiende o un formato de texto inesperado). Para intentar solucionarlo, el agente vuelve a llamar a la misma herramienta, recibe el mismo error, y lo intenta de nuevo.
Esto genera un ciclo infinito que tiene consecuencias críticas: consume rápidamente el saldo de la API (costes económicos muy elevados), satura el servidor (rate limits) y hace que la aplicación se quede cargando para siempre sin responderle al usuario.

Mitigaciones principales:

Límite de pasos (Max Iterations): Es el salvavidas más común y básico (LangChain lo incluye en su AgentExecutor). Consiste en configurar al agente con un número máximo de "pasos" permitidos por pregunta (por ejemplo, un máximo de 5 iteraciones). Si alcanza ese límite sin lograr su objetivo, el sistema lo aborta automáticamente y devuelve la mejor respuesta parcial que tenga o un mensaje de error controlado.

Detector de repetición: Es una medida de seguridad más inteligente. Se trata de un mecanismo que vigila el "Scratchpad" (el historial de pensamientos) del agente. Si detecta que el modelo está intentando ejecutar exactamente la misma herramienta con los mismos parámetros dos o tres veces seguidas, interviene para detener la ejecución o le inyecta un aviso al modelo para obligarlo a cambiar de estrategia.